# PySyft Researcher Workflow — Final Spatial Health Analysis

This notebook logs in as the restricted researcher, verifies that raw data access is blocked, develops analysis code against mock data, submits multiple code requests within one shared project, executes approved functions, and exports results as CSV plus GeoPackage / ESRI Shapefile layers.

No PNG outputs are produced.

## 0. Login as researcher

In [ ]:
import syft as sy

print("PySyft version:", sy.__version__)

ds = sy.login(
    url="http://127.0.0.1",
    port=8093,
    email="researcher@example.com",
    password="pw",
)

ds.refresh()
ds

## 1. Inspect datasets and assign assets

In [ ]:
ds.refresh()

for i, d in enumerate(ds.datasets):
    print("DATASET", i, d.name)
    for j, a in enumerate(d.assets):
        print("  ASSET", j, a.name)
        print("  MOCK COLUMNS:", a.mock.columns.tolist())
        print()

health_dataset = next(d for d in ds.datasets if getattr(d, "name", None) == "Health Distribution Dataset")
grid_dataset = next(d for d in ds.datasets if getattr(d, "name", None) == "Hungary Population Grid")

points_asset = health_dataset.assets[0]
grid_asset = grid_dataset.assets[0]

display(points_asset.mock.head())
display(grid_asset.mock.head())

## 2. Demonstrate that raw private data are inaccessible

In [ ]:
try:
    points_asset.data
except Exception as e:
    print("Raw point data access blocked as expected:")
    print(type(e).__name__, e)

try:
    grid_asset.data
except Exception as e:
    print("Raw grid data access blocked as expected:")
    print(type(e).__name__, e)

## 3. Local helper functions for mock-data tests

These helpers are only used locally on mock data. The Syft functions below are self-contained and repeat all required imports and reconstruction logic internally.

In [ ]:
def _rebuild_grid_gdf_local(grid, grid_crs="EPSG:23700"):
    import pandas as pd
    import geopandas as gpd
    from shapely import wkt

    GRID_ID_COL = "GID"
    POP_COL = "VALUE"

    required = {GRID_ID_COL, POP_COL, "geometry_wkt"}
    if not required.issubset(grid.columns):
        raise ValueError(f"Grid missing columns: {required - set(grid.columns)}. Available: {list(grid.columns)}")

    grid = grid[[GRID_ID_COL, POP_COL, "geometry_wkt"]].copy()
    grid[POP_COL] = pd.to_numeric(grid[POP_COL], errors="coerce").fillna(0)
    grid["geometry"] = grid["geometry_wkt"].apply(wkt.loads)
    return gpd.GeoDataFrame(grid.drop(columns="geometry_wkt"), geometry="geometry", crs=grid_crs)


def _rebuild_points_gdf_local(points, source_crs="EPSG:4326", target_crs="EPSG:23700"):
    import pandas as pd
    import geopandas as gpd
    from shapely.geometry import Point

    required = {"long", "lat"}
    if not required.issubset(points.columns):
        raise ValueError(f"Points missing columns: {required - set(points.columns)}. Available: {list(points.columns)}")

    points = points.copy()
    points["long"] = pd.to_numeric(points["long"], errors="coerce")
    points["lat"] = pd.to_numeric(points["lat"], errors="coerce")
    points = points.dropna(subset=["long", "lat"])

    return gpd.GeoDataFrame(
        points,
        geometry=[Point(x, y) for x, y in zip(points["long"], points["lat"])],
        crs=source_crs,
    ).to_crs(target_crs)


def _count_points_to_grid_local(points, grid):
    import geopandas as gpd
    GRID_ID_COL = "GID"
    points_gdf = _rebuild_points_gdf_local(points)
    grid_gdf = _rebuild_grid_gdf_local(grid)
    joined = gpd.sjoin(
        points_gdf[["geometry"]],
        grid_gdf[[GRID_ID_COL, "geometry"]],
        how="inner",
        predicate="within",
    )
    counts = joined.groupby(GRID_ID_COL).size().reset_index(name="Y")
    out = grid_gdf.merge(counts, on=GRID_ID_COL, how="left")
    out["Y"] = out["Y"].fillna(0).astype(int)
    return out, joined

## 4. Local mock-data checks

In [ ]:
import pandas as pd
import numpy as np

def visits_summary_local(x):
    df = x.copy()
    s = pd.to_numeric(df["VISITS"], errors="coerce")
    has = s.notna().any()

    return pd.DataFrame([{
        "variable": "VISITS",
        "n_rows": int(len(s)),
        "n_missing": int(s.isna().sum()),
        "missing_share": float(s.isna().mean()),
        "mean": float(s.mean()) if has else np.nan,
        "median": float(s.median()) if has else np.nan,
        "variance": float(s.var(ddof=1)) if has else np.nan,
        "min": float(s.min()) if has else np.nan,
        "max": float(s.max()) if has else np.nan,
        "p05": float(s.quantile(0.05)) if has else np.nan,
        "p25": float(s.quantile(0.25)) if has else np.nan,
        "p75": float(s.quantile(0.75)) if has else np.nan,
        "p95": float(s.quantile(0.95)) if has else np.nan,
    }])

visits_summary_local(points_asset.mock)

In [ ]:
mock_grid_counts, mock_joined = _count_points_to_grid_local(points_asset.mock, grid_asset.mock)
mock_grid_release = mock_grid_counts[(mock_grid_counts["Y"] > 5) & (mock_grid_counts["VALUE"] >= 50)].copy()

print("Mock joined points:", len(mock_joined))
print("Mock cells with points:", int((mock_grid_counts["Y"] > 0).sum()))
print("Mock release cells (Y > 5 and VALUE >= 50):", len(mock_grid_release))

mock_grid_release.head()

## 5. Define Syft function: VISITS summary

In [ ]:
@sy.syft_function_single_use(x=points_asset)
def visits_summary(x):
    import pandas as pd
    import numpy as np

    df = x.copy()

    if "VISITS" not in df.columns:
        raise ValueError(f"Column 'VISITS' not found. Available columns: {list(df.columns)}")

    s = pd.to_numeric(df["VISITS"], errors="coerce")
    has = s.notna().any()

    return pd.DataFrame([{
        "variable": "VISITS",
        "n_rows": int(len(s)),
        "n_missing": int(s.isna().sum()),
        "missing_share": float(s.isna().mean()),
        "mean": float(s.mean()) if has else np.nan,
        "median": float(s.median()) if has else np.nan,
        "variance": float(s.var(ddof=1)) if has else np.nan,
        "min": float(s.min()) if has else np.nan,
        "max": float(s.max()) if has else np.nan,
        "p05": float(s.quantile(0.05)) if has else np.nan,
        "p25": float(s.quantile(0.25)) if has else np.nan,
        "p75": float(s.quantile(0.75)) if has else np.nan,
        "p95": float(s.quantile(0.95)) if has else np.nan,
    }])

## 6. Define Syft function: disclosure-controlled descriptive grid release

In [ ]:
@sy.syft_function_single_use(points=points_asset, grid=grid_asset)
def create_published_grid_release(points, grid):
    import pandas as pd
    import geopandas as gpd
    from shapely import wkt
    from shapely.geometry import Point

    GRID_ID_COL = "GID"
    POP_COL = "VALUE"
    K_MIN = 5
    PMIN = 50
    GRID_CRS = "EPSG:23700"
    POINT_CRS = "EPSG:4326"

    grid = grid[[GRID_ID_COL, POP_COL, "geometry_wkt"]].copy()
    grid[POP_COL] = pd.to_numeric(grid[POP_COL], errors="coerce").fillna(0)
    grid["geometry"] = grid["geometry_wkt"].apply(wkt.loads)
    grid_gdf = gpd.GeoDataFrame(grid.drop(columns="geometry_wkt"), geometry="geometry", crs=GRID_CRS)

    points = points.copy()
    points["long"] = pd.to_numeric(points["long"], errors="coerce")
    points["lat"] = pd.to_numeric(points["lat"], errors="coerce")
    points = points.dropna(subset=["long", "lat"])
    points_gdf = gpd.GeoDataFrame(
        points,
        geometry=[Point(x, y) for x, y in zip(points["long"], points["lat"])],
        crs=POINT_CRS,
    ).to_crs(GRID_CRS)

    joined = gpd.sjoin(
        points_gdf[["geometry"]],
        grid_gdf[[GRID_ID_COL, "geometry"]],
        how="inner",
        predicate="within",
    )
    counts = joined.groupby(GRID_ID_COL).size().reset_index(name="Y")

    out = grid_gdf.merge(counts, on=GRID_ID_COL, how="left")
    out["Y"] = out["Y"].fillna(0).astype(int)
    out = out[(out["Y"] > K_MIN) & (out[POP_COL] >= PMIN)].copy()
    out["geometry_wkt"] = out.geometry.to_wkt()

    return pd.DataFrame(out[[GRID_ID_COL, POP_COL, "Y", "geometry_wkt"]])

## 7. Define Syft function: locally smoothed grid release

In [ ]:
@sy.syft_function_single_use(points=points_asset, grid=grid_asset)
def create_smoothed_grid_release(points, grid):
    import pandas as pd
    import numpy as np
    import geopandas as gpd
    from shapely import wkt
    from shapely.geometry import Point

    GRID_ID_COL = "GID"
    POP_COL = "VALUE"
    K_MIN = 5
    PMIN = 50
    GRID_CRS = "EPSG:23700"
    POINT_CRS = "EPSG:4326"

    grid = grid[[GRID_ID_COL, POP_COL, "geometry_wkt"]].copy()
    grid[POP_COL] = pd.to_numeric(grid[POP_COL], errors="coerce").fillna(0)
    grid["geometry"] = grid["geometry_wkt"].apply(wkt.loads)
    g = gpd.GeoDataFrame(grid.drop(columns="geometry_wkt"), geometry="geometry", crs=GRID_CRS)

    points = points.copy()
    points["long"] = pd.to_numeric(points["long"], errors="coerce")
    points["lat"] = pd.to_numeric(points["lat"], errors="coerce")
    points = points.dropna(subset=["long", "lat"])
    points_gdf = gpd.GeoDataFrame(
        points,
        geometry=[Point(x, y) for x, y in zip(points["long"], points["lat"])],
        crs=POINT_CRS,
    ).to_crs(GRID_CRS)

    joined = gpd.sjoin(points_gdf[["geometry"]], g[[GRID_ID_COL, "geometry"]], how="inner", predicate="within")
    counts = joined.groupby(GRID_ID_COL).size().reset_index(name="Y")

    g = g.merge(counts, on=GRID_ID_COL, how="left")
    g["Y"] = g["Y"].fillna(0).astype(int)

    # Queen-style local smoothing: mean Y of touching cells plus self.
    sindex = g.sindex
    Y = g["Y"].to_numpy(dtype=float)
    smooth = np.full(len(g), np.nan, dtype=float)
    geoms = list(g.geometry)

    for i, geom in enumerate(geoms):
        candidate_idx = list(sindex.intersection(geom.bounds))
        neigh = []
        for j in candidate_idx:
            geom_j = geoms[j]
            if geom.touches(geom_j) or geom.equals(geom_j):
                neigh.append(j)
        if i not in neigh:
            neigh.append(i)
        vals = Y[neigh]
        vals = vals[np.isfinite(vals)]
        smooth[i] = float(vals.mean()) if len(vals) else np.nan

    g["Y_smooth"] = smooth
    g["valid"] = (g["Y"] > K_MIN) & (g[POP_COL] >= PMIN)
    out = g[g["valid"]].copy()
    out["geometry_wkt"] = out.geometry.to_wkt()

    return pd.DataFrame(out[[GRID_ID_COL, POP_COL, "Y", "Y_smooth", "geometry_wkt"]])

## 8. Define Syft function: Global Moran's I

This implementation uses the populated analysis universe (`VALUE > 0`) before constructing Queen weights, matching the local-script behavior more closely than using all grid cells.

In [ ]:
@sy.syft_function_single_use(points=points_asset, grid=grid_asset)
def global_morans_i_grid(points, grid):
    import pandas as pd
    import geopandas as gpd
    from shapely import wkt
    from shapely.geometry import Point
    from libpysal.weights import Queen
    from esda.moran import Moran

    GRID_ID_COL = "GID"
    POP_COL = "VALUE"
    GRID_CRS = "EPSG:23700"
    POINT_CRS = "EPSG:4326"
    N_PERMUTATIONS = 999

    grid = grid[[GRID_ID_COL, POP_COL, "geometry_wkt"]].copy()
    grid[POP_COL] = pd.to_numeric(grid[POP_COL], errors="coerce").fillna(0)
    grid["geometry"] = grid["geometry_wkt"].apply(wkt.loads)
    grid_gdf = gpd.GeoDataFrame(grid.drop(columns="geometry_wkt"), geometry="geometry", crs=GRID_CRS)

    points = points.copy()
    points["long"] = pd.to_numeric(points["long"], errors="coerce")
    points["lat"] = pd.to_numeric(points["lat"], errors="coerce")
    points = points.dropna(subset=["long", "lat"])
    points_gdf = gpd.GeoDataFrame(
        points,
        geometry=[Point(x, y) for x, y in zip(points["long"], points["lat"])],
        crs=POINT_CRS,
    ).to_crs(GRID_CRS)

    joined = gpd.sjoin(points_gdf[["geometry"]], grid_gdf[[GRID_ID_COL, "geometry"]], how="inner", predicate="within")
    counts = joined.groupby(GRID_ID_COL).size().reset_index(name="Y")

    g = grid_gdf.merge(counts, on=GRID_ID_COL, how="left")
    g["Y"] = g["Y"].fillna(0).astype(float)

    # Match original analysis universe: populated grid cells only.
    g = g[g[POP_COL] > 0].reset_index(drop=True).copy()

    w = Queen.from_dataframe(g, use_index=False)
    w.transform = "r"
    moran = Moran(g["Y"].to_numpy(), w, permutations=N_PERMUTATIONS)

    return pd.DataFrame([{
        "variable": "Y",
        "n_points_joined": int(len(joined)),
        "n_cells": int(len(g)),
        "n_nonzero_cells": int((g["Y"] > 0).sum()),
        "moran_I": float(moran.I),
        "expected_I": float(moran.EI),
        "p_norm": float(moran.p_norm),
        "p_sim": float(moran.p_sim),
        "z_norm": float(moran.z_norm),
        "z_sim": float(moran.z_sim),
        "permutations": int(N_PERMUTATIONS),
    }])

## 9. Define Syft function: Local Moran's I / LISA

In [ ]:
@sy.syft_function_single_use(points=points_asset, grid=grid_asset)
def local_morans_i_grid(points, grid):
    import pandas as pd
    import geopandas as gpd
    from shapely import wkt
    from shapely.geometry import Point
    from libpysal.weights import Queen
    from esda.moran import Moran_Local

    GRID_ID_COL = "GID"
    POP_COL = "VALUE"
    GRID_CRS = "EPSG:23700"
    POINT_CRS = "EPSG:4326"
    N_PERMUTATIONS = 999
    P_THRESHOLD = 0.05

    grid = grid[[GRID_ID_COL, POP_COL, "geometry_wkt"]].copy()
    grid[POP_COL] = pd.to_numeric(grid[POP_COL], errors="coerce").fillna(0)
    grid["geometry"] = grid["geometry_wkt"].apply(wkt.loads)
    grid_gdf = gpd.GeoDataFrame(grid.drop(columns="geometry_wkt"), geometry="geometry", crs=GRID_CRS)

    points = points.copy()
    points["long"] = pd.to_numeric(points["long"], errors="coerce")
    points["lat"] = pd.to_numeric(points["lat"], errors="coerce")
    points = points.dropna(subset=["long", "lat"])
    points_gdf = gpd.GeoDataFrame(
        points,
        geometry=[Point(x, y) for x, y in zip(points["long"], points["lat"])],
        crs=POINT_CRS,
    ).to_crs(GRID_CRS)

    joined = gpd.sjoin(points_gdf[["geometry"]], grid_gdf[[GRID_ID_COL, "geometry"]], how="inner", predicate="within")
    counts = joined.groupby(GRID_ID_COL).size().reset_index(name="Y")

    g = grid_gdf.merge(counts, on=GRID_ID_COL, how="left")
    g["Y"] = g["Y"].fillna(0).astype(float)

    # Match original analysis universe: populated grid cells only.
    g = g[g[POP_COL] > 0].reset_index(drop=True).copy()

    w = Queen.from_dataframe(g, use_index=False)
    w.transform = "r"
    lisa = Moran_Local(g["Y"].to_numpy(), w, permutations=N_PERMUTATIONS)

    g["lisa_i"] = lisa.Is
    g["lisa_p"] = lisa.p_sim
    g["lisa_q"] = lisa.q
    g["sig"] = g["lisa_p"] < P_THRESHOLD
    g["cluster"] = "Not significant"
    g.loc[(g["lisa_q"] == 1) & (g["sig"]), "cluster"] = "High-High"
    g.loc[(g["lisa_q"] == 2) & (g["sig"]), "cluster"] = "Low-High"
    g.loc[(g["lisa_q"] == 3) & (g["sig"]), "cluster"] = "Low-Low"
    g.loc[(g["lisa_q"] == 4) & (g["sig"]), "cluster"] = "High-Low"
    g["geometry_wkt"] = g.geometry.to_wkt()

    return pd.DataFrame(g[[GRID_ID_COL, POP_COL, "Y", "lisa_i", "lisa_p", "lisa_q", "sig", "cluster", "geometry_wkt"]])

## 10. Define Syft function: Kernel Density Estimation

In [ ]:
@sy.syft_function_single_use(points=points_asset, grid=grid_asset)
def kernel_density_surface(points, grid):
    import pandas as pd
    import numpy as np
    import geopandas as gpd
    from shapely import wkt
    from shapely.geometry import Point
    from sklearn.neighbors import KernelDensity

    GRID_ID_COL = "GID"
    POP_COL = "VALUE"
    GRID_CRS = "EPSG:23700"
    POINT_CRS = "EPSG:4326"
    CELL_SIZE = 100
    BANDWIDTH = 500

    grid = grid[[GRID_ID_COL, POP_COL, "geometry_wkt"]].copy()
    grid["geometry"] = grid["geometry_wkt"].apply(wkt.loads)
    grid_gdf = gpd.GeoDataFrame(grid.drop(columns="geometry_wkt"), geometry="geometry", crs=GRID_CRS)

    points = points.copy()
    points["long"] = pd.to_numeric(points["long"], errors="coerce")
    points["lat"] = pd.to_numeric(points["lat"], errors="coerce")
    points = points.dropna(subset=["long", "lat"])
    points_gdf = gpd.GeoDataFrame(
        points,
        geometry=[Point(x, y) for x, y in zip(points["long"], points["lat"])],
        crs=POINT_CRS,
    ).to_crs(GRID_CRS)

    coords = np.vstack([points_gdf.geometry.x, points_gdf.geometry.y]).T
    xmin, ymin, xmax, ymax = grid_gdf.total_bounds
    xgrid = np.arange(xmin, xmax, CELL_SIZE)
    ygrid = np.arange(ymin, ymax, CELL_SIZE)
    xx, yy = np.meshgrid(xgrid, ygrid)
    grid_coords = np.vstack([xx.ravel(), yy.ravel()]).T

    kde = KernelDensity(bandwidth=BANDWIDTH, kernel="gaussian")
    kde.fit(coords)
    density = np.exp(kde.score_samples(grid_coords))

    return pd.DataFrame({
        "x": grid_coords[:, 0].astype(float),
        "y": grid_coords[:, 1].astype(float),
        "density": density.astype(float),
        "cell_size": float(CELL_SIZE),
        "bandwidth": float(BANDWIDTH),
        "crs": GRID_CRS,
    })

## 11. Create or retrieve one shared project

In [ ]:
PROJECT_NAME = "Health Spatial Privacy Analysis"

ds.refresh()
existing = [p for p in ds.projects if getattr(p, "name", None) == PROJECT_NAME]

if existing:
    proj = existing[0]
    print("Using existing project:", PROJECT_NAME)
else:
    project = sy.Project(
        name=PROJECT_NAME,
        description=(
            "Privacy-preserving spatial health analysis using the protected "
            "point dataset and population grid. Includes summary statistics, "
            "descriptive grid release, smoothed mapping, Global Moran's I, "
            "Local Moran's I, and KDE."
        ),
        members=[ds],
    )
    proj = project.send()
    print("Created project:", PROJECT_NAME)

proj

## 12. Submit all code requests

This cell is safe to rerun. If a request already exists for a function, it is skipped.

In [ ]:
functions_to_request = [
    ("visits_summary", visits_summary),
    ("create_published_grid_release", create_published_grid_release),
    ("create_smoothed_grid_release", create_smoothed_grid_release),
    ("global_morans_i_grid", global_morans_i_grid),
    ("local_morans_i_grid", local_morans_i_grid),
    ("kernel_density_surface", kernel_density_surface),
]

for name, func in functions_to_request:
    try:
        res = proj.create_code_request(func)
        print(f"Submitted request for {name}:", res)
    except Exception as e:
        msg = str(e)
        if "already exists for this UserCode" in msg:
            print(f"Request already exists for {name}; skipping.")
        else:
            print(f"Failed request for {name}: {type(e).__name__}: {e}")
            raise

## 13. After owner approval: inspect approved code

In [ ]:
ds.refresh()
dir(ds.code)

## 14. Execute approved analyses

In [ ]:
summary_result = ds.code.visits_summary(x=points_asset).get()
summary_result

In [ ]:
grid_result = ds.code.create_published_grid_release(points=points_asset, grid=grid_asset).get()
grid_result.head(), grid_result.shape

In [ ]:
smoothed_result = ds.code.create_smoothed_grid_release(points=points_asset, grid=grid_asset).get()
smoothed_result.head(), smoothed_result.shape

In [ ]:
global_moran_result = ds.code.global_morans_i_grid(points=points_asset, grid=grid_asset).get()
global_moran_result

In [ ]:
local_moran_result = ds.code.local_morans_i_grid(points=points_asset, grid=grid_asset).get()
local_moran_result.head(), local_moran_result.shape

In [ ]:
kde_result = ds.code.kernel_density_surface(points=points_asset, grid=grid_asset).get()
kde_result.head(), kde_result.shape

## 15. Export tabular CSV results

All outputs are exported as CSV. Spatial outputs are additionally exported below as GeoPackage and ESRI Shapefile layers.

In [ ]:
import os

os.makedirs("results", exist_ok=True)

summary_result.to_csv("results/visits_summary.csv", index=False)
grid_result.to_csv("results/published_grid_release.csv", index=False)
smoothed_result.to_csv("results/smoothed_grid_release.csv", index=False)
global_moran_result.to_csv("results/global_morans_i.csv", index=False)
local_moran_result.to_csv("results/local_morans_i.csv", index=False)
kde_result.to_csv("results/kernel_density_surface.csv", index=False)

print("CSV results exported to ./results")

## 16. Export spatial outputs as GeoPackage and ESRI Shapefile

GeoPackage is exported for robustness. ESRI Shapefile is exported for compatibility with older GIS workflows. Note that Shapefile creates multiple sidecar files and truncates field names to 10 characters, so short field names are used for Shapefile layers.

In [ ]:
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point

GRID_CRS = "EPSG:23700"

def export_polygon_result(df, filename_base):
    out = df.copy()
    out["geometry"] = out["geometry_wkt"].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(out.drop(columns="geometry_wkt"), geometry="geometry", crs=GRID_CRS)

    # GeoPackage: full field names supported.
    gdf.to_file(f"results/{filename_base}.gpkg", driver="GPKG")

    # Shapefile: use short field names to avoid truncation/encoding surprises.
    shp = gdf.copy()
    shp = shp.rename(columns={
        "VALUE": "VALUE",
        "Y_smooth": "Ysmooth",
        "lisa_i": "lisa_i",
        "lisa_p": "lisa_p",
        "lisa_q": "lisa_q",
        "sig": "sig",
        "cluster": "cluster",
    })
    shp.to_file(f"results/{filename_base}.shp", driver="ESRI Shapefile")
    return gdf

published_gdf = export_polygon_result(grid_result, "published_grid_release")
smoothed_gdf = export_polygon_result(smoothed_result, "smoothed_grid_release")
local_moran_gdf = export_polygon_result(local_moran_result, "local_morans_i")

# KDE output as point layer.
kde_gdf = gpd.GeoDataFrame(
    kde_result.copy(),
    geometry=[Point(x, y) for x, y in zip(kde_result["x"], kde_result["y"])],
    crs=GRID_CRS,
)
kde_gdf.to_file("results/kernel_density_surface_points.gpkg", driver="GPKG")
kde_gdf.to_file("results/kernel_density_surface_points.shp", driver="ESRI Shapefile")

print("Spatial results exported to ./results as GeoPackage and ESRI Shapefile layers.")

## 17. Export summary of generated files

In [ ]:
import os
for f in sorted(os.listdir("results")):
    print(f)